# 02 - Preprocessing Bahasa Indonesia

Notebook ini memindahkan pipeline preprocessing dari `src/preprocessing.py` ke sel yang dapat dilihat dan dijalankan: cleaning, case folding, normalisasi slang, stopword removal, dan stemming Sastrawi pada contoh teks. Corpus utama menggunakan preprocessing stabil agar notebook selesai pada dataset besar.

In [1]:
import re
import string
from pathlib import Path
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [2]:
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "raw").is_dir())
RAW_PATH = ROOT / "data" / "raw" / "coretax_reviews.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "coretax_reviews_clean.csv"

SLANG_NORMALIZATION = {"bgt": "banget", "ga": "tidak", "gak": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak", "aja": "saja", "apk": "aplikasi", "blm": "belum", "belom": "belum", "bener": "benar", "krn": "karena", "lg": "lagi", "msh": "masih", "ribet": "rumit", "udh": "sudah", "udah": "sudah", "yg": "yang"}
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\\S+|www\\S+|@\\w+|#\\w+", " ", text)
    text = re.sub(r"[0-9]+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"[^a-z\\s]", " ", text)
    text = re.sub(r"(.)\\1{2,}", r"\\1\\1", text)
    return re.sub(r"\\s+", " ", text).strip()[:2000]

def normalize_slang(text):
    return " ".join(SLANG_NORMALIZATION.get(token, token) for token in text.split())

def preprocess(text):
    normalized = normalize_slang(clean_text(text))
    return stemmer.stem(stopword_remover.remove(normalized))

sample = "APK-nya ribet bgt, gak bisa login!"
print("Sebelum:", sample)
print("Sesudah:", preprocess(sample))

if PROCESSED_PATH.exists():
    df = pd.read_csv(PROCESSED_PATH)
    df = df.rename(columns={"content": "review_text", "score": "rating"})
    print(f"Memuat hasil preprocessing: {len(df):,} baris")
else:
    df = pd.read_csv(RAW_PATH)
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    df = df.dropna(subset=["review_text", "rating"])
    df = df[df["rating"].between(1, 5)].copy()
    df["review_text"] = df["review_text"].astype(str).str.strip()
    df = df[df["review_text"].ne("")].drop_duplicates("review_text").copy()
    df["sentiment"] = df["rating"].map(lambda rating: "negatif" if rating <= 2 else ("netral" if rating == 3 else "positif"))
    df["clean_text"] = df["review_text"].map(lambda text: stopword_remover.remove(normalize_slang(clean_text(text))))
    df = df[df["clean_text"].str.split().str.len().ge(1)].copy()
    df["text_length"] = df["clean_text"].str.split().str.len()
    PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED_PATH, index=False)
    print(f"Data tersimpan: {PROCESSED_PATH}")

df[["review_text", "clean_text", "sentiment"]].head()

Sebelum: APK-nya ribet bgt, gak bisa login!
Sesudah: apknya rumit banget bisa login
Memuat hasil preprocessing: 7,857 baris


,review_text,clean_text,sentiment
0,"aplikasi sampah,",aplikasi sampah,negatif
1,mau login malah ribet banget udah pake duit ra...,mau login malah rumit banget pake duit rakyat ...,negatif
2,app murahan,app murahan,negatif
3,Data udah betul semua✌️✌️ tidak bisa daftar 👍👍...,data betul semua bisa daftar katanya status pe...,positif
4,"Astaghfirullah.,., ampunilah kami ya Allah..",astaghfirullah ampunilah ya allah,negatif
